In [5]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
  llm=llm,
  max_token_limit=120,
  memory_key="chat_history",
  return_messages=True
)

prompt = ChatPromptTemplate.from_messages([
  ("system", "You are a helpful AI talking to a human"),
  MessagesPlaceholder(variable_name="chat_history"),
  ("human", "{question}")
])

def load_memory(_):
  return memory.load_memory_variables({})["chat_history"]

chain = RunnablePassthrough.assign(chat_history=load_memory) | prompt | llm

def invoke_chain(question):
  result = chain.invoke({"question" : question})
  memory.save_context({"input": question}, {"output": result.content})
  print(result)
  
invoke_chain("My name is Yongjae")


content='Nice to meet you, Yongjae! How can I assist you today?'


In [6]:
invoke_chain("I live in New York")

content="That's great! New York is a vibrant and exciting city. Is there anything specific you'd like to know or talk about regarding New York?"


In [7]:
invoke_chain("What is my name?")

content='Your name is Yongjae. How can I assist you further, Yongjae?'


In [8]:
memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='My name is Yongjae'),
  AIMessage(content='Nice to meet you, Yongjae! How can I assist you today?'),
  HumanMessage(content='I live in New York'),
  AIMessage(content="That's great! New York is a vibrant and exciting city. Is there anything specific you'd like to know or talk about regarding New York?"),
  HumanMessage(content='What is my name?'),
  AIMessage(content='Your name is Yongjae. How can I assist you further, Yongjae?')]}